# Marjum 2026-07 — Joint MCMC: Per-Camera Review

One page per camera. For each: its seven sampled parameters with all 8 chains
overlaid, its own posterior E/N cloud on a hillshade crop, and the spread of its
predicted horizon across posterior draws against the actual segmentation.

Panel titles carry that coordinate's Rhat, red when it misses the 1.01
threshold — which, in this run, is nearly everywhere. Companion document:
the summary review.

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

warnings.simplefilter('ignore', RuntimeWarning)
WORKSPACE = Path('/home/aparsons/projects/eigsep/terrain')
RUN = WORKSPACE / 'joint_posterior_v1'

manifest = json.loads((RUN / 'manifest.json').read_text())
conv = json.loads((RUN / 'convergence.json').read_text())
with np.load(RUN / 'combined.npz', allow_pickle=True) as z:
    draws = np.asarray(z['draws'], float)
    rhat = np.asarray(z['rhat'], float)
    ess = np.asarray(z['ess_bulk'], float)
    names = [str(s) for s in z['names']]
    keys = [str(s) for s in z['keys']]
nchain, ndraw, nparam = draws.shape
PRM = ('e', 'n', 'u', 'th', 'ph', 'ti', 'logf')
flat = draws.reshape(-1, nparam)
HR = WORKSPACE / 'horizon_range'
hr_summary = json.loads((HR / 'summary.json').read_text()) if (HR / 'summary.json').exists() else {}

import matplotlib.image as mpimg
from eigsep_terrain.marjum_dem import MarjumDEM as DEM
from marjum_bundle import working_grid
from terrain_hillshade_cmap import plot_terrain_hillshade

dem = working_grid(DEM(cache_file='marjum_dem_sw.npz'))
se, sn = dem._working_shift
dem_raw = DEM(cache_file='marjum_dem_sw.npz')
print('cameras:', len(keys))

In [ ]:
def camera_page(key):
    b = keys.index(key) * 7
    fig = plt.figure(figsize=(11.5, 8.6))
    gs = fig.add_gridspec(3, 7, height_ratios=[1.0, 1.5, 0.06], hspace=.45, wspace=.35)

    for j, prm in enumerate(PRM):
        ax = fig.add_subplot(gs[0, j])
        for c in range(nchain):
            ax.plot(draws[c, ::max(1, ndraw // 600), b + j], lw=.35, alpha=.75, rasterized=True)
        ax.set_xticks([]); ax.tick_params(labelsize=5)
        bad = rhat[b + j] > 1.01
        ax.set_title(f'{prm}\nR={rhat[b+j]:.2f} ESS={ess[b+j]:.0f}', fontsize=6,
                     color='firebrick' if bad else 'black')

    axm = fig.add_subplot(gs[1, :3])
    e, n = flat[:, b], flat[:, b + 1]
    pad = max(12.0, 4 * max(e.std(), n.std()))
    e0, n0 = e.mean(), n.mean()
    E, N, U = dem_raw.get_tile(erng_m=(e0 - se - pad, e0 - se + pad),
                               nrng_m=(n0 - sn - pad, n0 - sn + pad), mesh=False)
    E, N = E + se, N + sn
    plot_terrain_hillshade(axm, U, res=float(dem.res), extent=(E[0], E[-1], N[0], N[-1]))
    TH = max(1, len(e) // 3000)
    axm.scatter(e[::TH], n[::TH], s=2, color='red', alpha=.4, linewidths=0, rasterized=True)
    axm.plot(e0, n0, 'w*', ms=13, mec='k')
    axm.set_xlabel('East [m]', fontsize=7); axm.set_ylabel('North [m]', fontsize=7)
    axm.tick_params(labelsize=6)
    axm.set_title(f'E/N draws  (sd {e.std():.2f} / {n.std():.2f} m)', fontsize=8)

    axh = fig.add_subplot(gs[1, 3:])
    p = WORKSPACE / 'horizon_range' / f'horizon_range_{key}.png'
    if p.exists():
        axh.imshow(mpimg.imread(str(p)))
    else:
        axh.text(.5, .5, 'horizon panel not rendered', ha='center')
    axh.axis('off')

    hv = hr_summary.get(key, {})
    fig.suptitle(f'IMG_{key}   —   focal ESS {ess[b+6]:.0f}, Rhat {rhat[b+6]:.2f}   |   '
                 f"horizon spread {hv.get('median_horizon_spread_px', float('nan')):.0f} px median",
                 fontsize=11, y=.995)
    plt.show()
    plt.close(fig)

order = sorted(keys, key=lambda k: ess[keys.index(k) * 7 + 6])
print('pages ordered worst-mixing focal length first:', order[:5], '...')

In [ ]:
if 0 < len(order):
    camera_page(order[0])

In [ ]:
if 1 < len(order):
    camera_page(order[1])

In [ ]:
if 2 < len(order):
    camera_page(order[2])

In [ ]:
if 3 < len(order):
    camera_page(order[3])

In [ ]:
if 4 < len(order):
    camera_page(order[4])

In [ ]:
if 5 < len(order):
    camera_page(order[5])

In [ ]:
if 6 < len(order):
    camera_page(order[6])

In [ ]:
if 7 < len(order):
    camera_page(order[7])

In [ ]:
if 8 < len(order):
    camera_page(order[8])

In [ ]:
if 9 < len(order):
    camera_page(order[9])

In [ ]:
if 10 < len(order):
    camera_page(order[10])

In [ ]:
if 11 < len(order):
    camera_page(order[11])

In [ ]:
if 12 < len(order):
    camera_page(order[12])

In [ ]:
if 13 < len(order):
    camera_page(order[13])

In [ ]:
if 14 < len(order):
    camera_page(order[14])

In [ ]:
if 15 < len(order):
    camera_page(order[15])

In [ ]:
if 16 < len(order):
    camera_page(order[16])

In [ ]:
if 17 < len(order):
    camera_page(order[17])

In [ ]:
if 18 < len(order):
    camera_page(order[18])

In [ ]:
if 19 < len(order):
    camera_page(order[19])

In [ ]:
if 20 < len(order):
    camera_page(order[20])

In [ ]:
if 21 < len(order):
    camera_page(order[21])

In [ ]:
if 22 < len(order):
    camera_page(order[22])

In [ ]:
if 23 < len(order):
    camera_page(order[23])

In [ ]:
if 24 < len(order):
    camera_page(order[24])

In [ ]:
if 25 < len(order):
    camera_page(order[25])

In [ ]:
if 26 < len(order):
    camera_page(order[26])

In [ ]:
if 27 < len(order):
    camera_page(order[27])

In [ ]:
if 28 < len(order):
    camera_page(order[28])